# Data Profiling & Initial Analysis
**Phase 1.1.1 - Step 1.1.2: Sample Inspection & Statistical Analysis**

**Date**: November 10, 2025  
**Objective**: Understand the structure and quality of scraped data

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")

## 1. File Paths Setup

In [ ]:
# Define paths
DATA_DIR = Path('../dataset')
SCRAPED_DIR = DATA_DIR / 'scraped'
REPORTS_DIR = Path('../reports')

# File paths
enhanced_path = SCRAPED_DIR / 'songs_enhanced_full.csv'
failed_path = SCRAPED_DIR / 'failed_tracks.csv'
unknown_path = SCRAPED_DIR / 'unknown_tracks.csv'
genre_map_path = SCRAPED_DIR / 'genre_mappings.csv'

# Verify files exist
for path in [enhanced_path, failed_path, unknown_path, genre_map_path]:
    if path.exists():
        print(f"✓ Found: {path.name} ({path.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"✗ Missing: {path.name}")

## 2. Sample Inspection - songs_enhanced_full.csv

In [ ]:
# Load sample
print("Loading first 1000 rows of songs_enhanced_full.csv...")
enhanced_sample = pd.read_csv(enhanced_path, nrows=1000)

print(f"\nShape: {enhanced_sample.shape}")
print(f"Columns: {len(enhanced_sample.columns)}")
print(f"\nColumn Names:")
print(enhanced_sample.columns.tolist())

In [ ]:
# Data types
print("Data Types:")
print(enhanced_sample.dtypes)

In [ ]:
# First few rows
enhanced_sample.head(10)

In [ ]:
# Basic info
enhanced_sample.info()

In [ ]:
# Missing values in sample
print("Missing values in sample:")
missing = enhanced_sample.isnull().sum()
missing_pct = (missing / len(enhanced_sample) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

## 3. Sample Inspection - failed_tracks.csv

In [ ]:
# Load failed tracks sample
print("Loading first 1000 rows of failed_tracks.csv...")
failed_sample = pd.read_csv(failed_path, nrows=1000)

print(f"\nShape: {failed_sample.shape}")
print(f"Columns: {failed_sample.columns.tolist()}")
failed_sample.head()

## 4. Sample Inspection - unknown_tracks.csv

In [ ]:
# Load unknown tracks sample
print("Loading first 1000 rows of unknown_tracks.csv...")
unknown_sample = pd.read_csv(unknown_path, nrows=1000)

print(f"\nShape: {unknown_sample.shape}")
print(f"Columns: {unknown_sample.columns.tolist()}")
unknown_sample.head()

## 5. Full Dataset Statistics (Chunked Processing)
**This section processes the entire dataset to get accurate statistics**

In [ ]:
# Statistical analysis function
def analyze_large_csv(filepath, chunksize=50000):
    """
    Analyze large CSV file in chunks
    """
    from tqdm.notebook import tqdm
    
    stats = {
        'total_rows': 0,
        'unique_ids': set(),
        'missing_values': {},
        'genre_issues': {'nan': 0, 'empty': 0},
        'year_issues': {'zero': 0, 'invalid': 0},
    }
    
    # First, get total lines for progress bar
    print(f"Counting lines in {filepath.name}...")
    total_lines = sum(1 for _ in open(filepath)) - 1
    
    # Process chunks
    print(f"Processing {total_lines:,} rows in chunks of {chunksize:,}...")
    
    with tqdm(total=total_lines) as pbar:
        for chunk in pd.read_csv(filepath, chunksize=chunksize):
            stats['total_rows'] += len(chunk)
            
            # Unique IDs (assuming track_id or similar column exists)
            if 'track_id' in chunk.columns:
                stats['unique_ids'].update(chunk['track_id'].unique())
            
            # Missing values
            for col in chunk.columns:
                if col not in stats['missing_values']:
                    stats['missing_values'][col] = 0
                stats['missing_values'][col] += chunk[col].isna().sum()
            
            # Genre issues
            if 'genre' in chunk.columns:
                stats['genre_issues']['nan'] += chunk['genre'].isna().sum()
                stats['genre_issues']['empty'] += (chunk['genre'] == '').sum()
            
            # Year issues
            if 'year' in chunk.columns:
                stats['year_issues']['zero'] += (chunk['year'] == 0).sum()
                stats['year_issues']['invalid'] += (
                    (chunk['year'] < 1900) | (chunk['year'] > 2025)
                ).sum()
            
            pbar.update(len(chunk))
    
    stats['unique_tracks'] = len(stats['unique_ids'])
    del stats['unique_ids']  # Remove set to save memory
    
    return stats

print("✓ Analysis function defined")

In [ ]:
# Analyze songs_enhanced_full.csv
print("Analyzing songs_enhanced_full.csv...\n")
enhanced_stats = analyze_large_csv(enhanced_path, chunksize=50000)

In [ ]:
# Display statistics
print("=" * 60)
print("SONGS_ENHANCED_FULL.CSV STATISTICS")
print("=" * 60)
print(f"\nTotal Rows: {enhanced_stats['total_rows']:,}")
print(f"Unique Tracks: {enhanced_stats['unique_tracks']:,}")
print(f"Duplicates: {enhanced_stats['total_rows'] - enhanced_stats['unique_tracks']:,}")

print("\n" + "-" * 60)
print("GENRE ISSUES:")
print("-" * 60)
print(f"NaN genres: {enhanced_stats['genre_issues']['nan']:,}")
print(f"Empty genres: {enhanced_stats['genre_issues']['empty']:,}")
total_genre_issues = enhanced_stats['genre_issues']['nan'] + enhanced_stats['genre_issues']['empty']
print(f"Total genre issues: {total_genre_issues:,} ({total_genre_issues/enhanced_stats['total_rows']*100:.2f}%)")

print("\n" + "-" * 60)
print("YEAR ISSUES:")
print("-" * 60)
print(f"Year = 0: {enhanced_stats['year_issues']['zero']:,}")
print(f"Invalid years (<1900 or >2025): {enhanced_stats['year_issues']['invalid']:,}")
total_year_issues = enhanced_stats['year_issues']['zero'] + enhanced_stats['year_issues']['invalid']
print(f"Total year issues: {total_year_issues:,} ({total_year_issues/enhanced_stats['total_rows']*100:.2f}%)")

print("\n" + "-" * 60)
print("MISSING VALUES BY COLUMN:")
print("-" * 60)
missing_df = pd.DataFrame.from_dict(enhanced_stats['missing_values'], orient='index', columns=['Count'])
missing_df['Percentage'] = (missing_df['Count'] / enhanced_stats['total_rows'] * 100).round(2)
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)
print(missing_df)

## 6. Generate Report File

In [ ]:
# Generate comprehensive report
from datetime import datetime

report_path = REPORTS_DIR / 'data_quality_report.txt'

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("DATA QUALITY REPORT - songs_enhanced_full.csv\n")
    f.write("=" * 80 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"File: {enhanced_path}\n")
    f.write(f"File Size: {enhanced_path.stat().st_size / 1e9:.2f} GB\n")
    f.write("\n")
    
    f.write("-" * 80 + "\n")
    f.write("DATASET OVERVIEW\n")
    f.write("-" * 80 + "\n")
    f.write(f"Total Rows: {enhanced_stats['total_rows']:,}\n")
    f.write(f"Unique Tracks: {enhanced_stats['unique_tracks']:,}\n")
    f.write(f"Duplicate Rows: {enhanced_stats['total_rows'] - enhanced_stats['unique_tracks']:,}\n")
    f.write("\n")
    
    f.write("-" * 80 + "\n")
    f.write("DATA QUALITY ISSUES\n")
    f.write("-" * 80 + "\n")
    f.write(f"\nGenre Issues:\n")
    f.write(f"  - NaN genres: {enhanced_stats['genre_issues']['nan']:,}\n")
    f.write(f"  - Empty genres: {enhanced_stats['genre_issues']['empty']:,}\n")
    f.write(f"  - Total: {total_genre_issues:,} ({total_genre_issues/enhanced_stats['total_rows']*100:.2f}%)\n")
    f.write(f"\nYear Issues:\n")
    f.write(f"  - Year = 0: {enhanced_stats['year_issues']['zero']:,}\n")
    f.write(f"  - Invalid years: {enhanced_stats['year_issues']['invalid']:,}\n")
    f.write(f"  - Total: {total_year_issues:,} ({total_year_issues/enhanced_stats['total_rows']*100:.2f}%)\n")
    f.write("\n")
    
    f.write("-" * 80 + "\n")
    f.write("MISSING VALUES BY COLUMN\n")
    f.write("-" * 80 + "\n")
    for col, row in missing_df.iterrows():
        f.write(f"{col:30s} {row['Count']:>10,} ({row['Percentage']:>6.2f}%)\n")
    f.write("\n")
    
    f.write("-" * 80 + "\n")
    f.write("RECOMMENDATIONS\n")
    f.write("-" * 80 + "\n")
    f.write("1. Address genre issues (NaN/empty values)\n")
    f.write("2. Fix year = 0 values\n")
    f.write("3. Remove duplicate rows\n")
    f.write("4. Validate all features against expected ranges\n")
    f.write("5. Document cleaning decisions\n")
    f.write("=" * 80 + "\n")

print(f"✓ Report saved to: {report_path}")
print("\nYou can now proceed to Phase 1.1.3 - Failed Tracks Analysis")

## 7. Next Steps

Based on this analysis:

1. ✅ **Completed**: Statistical analysis of songs_enhanced_full.csv
2. ⏭️ **Next**: Analyze failed_tracks.csv (Step 1.1.3)
3. ⏭️ **Then**: Analyze unknown_tracks.csv (Step 1.1.4)
4. ⏭️ **Then**: Define cleaning strategies (Phase 1.2)

**Key Decisions Needed**:
- How to handle NaN genres? (Drop / Impute / "Unknown")
- How to handle year = 0? (Drop / Impute / Flag)
- How to handle duplicates? (Keep first / Most complete)
- What to do with failed tracks?
- What to do with unknown genre tracks?

See `ml/DATA_VALIDATION_ROADMAP.md` for detailed guidance.